In [19]:
import pandas as pd, json, os, re, zipfile, pathlib
from pathlib import Path
excel_path = r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\ListOfNames.xlsx"
site_dir   = r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP"
os.makedirs(site_dir, exist_ok=True)

googlesheet = 'https://docs.google.com/spreadsheets/d/19IE4FLYkJ1MAvawhhZx5WEnj0xowtOIoqPIB-E-6rSU/edit?usp=sharing'
sheet_id = "19IE4FLYkJ1MAvawhhZx5WEnj0xowtOIoqPIB-E-6rSU"
gid = 0  # change if the tab isn't the first one
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df = pd.read_csv(url)
df['Name in Farsi'] = df['Name in Farsi'].str.strip()
#------------------------
#make sure all the names have their folders 
PHOTOS_DIR = Path(r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\Photos")
# Make sure base Photos directory exists
PHOTOS_DIR.mkdir(exist_ok=True)
def safe_folder_name(name: str) -> str:
    """
    Make Windows-safe folder names.
    Removes invalid characters.
    """
    invalid_chars = '<>:"/\\|?*'
    for ch in invalid_chars:
        name = name.replace(ch, "")
    return name.strip().rstrip(".")

def ensure_folders_exist(df):
    created = []
    skipped = 0
    for name in df["Name in English"]:
        if pd.isna(name):
            skipped += 1
            continue
        clean_name = safe_folder_name(str(name))
        if not clean_name:
            skipped += 1
            continue
        person_dir = PHOTOS_DIR / clean_name
        if not person_dir.exists():
            person_dir.mkdir(parents=True, exist_ok=True)
            created.append(clean_name)

    print("Done.")
    print(f"Folders created: {len(created)}")
    print(f"Rows skipped (blank/invalid): {skipped}")

    if created:
        print("\nNew folders:")
        for c in created[:20]:
            print(" -", c)
        if len(created) > 20:
            print("...")

# Run it
ensure_folders_exist(df)
#-------------------------
#---- Make Photos Column
# Root photos folder (contains subfolders named exactly like df["Name in English"])
PHOTOS_DIR = Path(r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\Photos")
# File extensions to include
EXTS = {".png", ".jpg", ".jpeg"}
def build_photos_cell(name_in_english: str) -> str:
    """
    For a given 'Name in English', look inside:
      PHOTOS_DIR / <Name in English> /
    and return a comma-separated string of relative paths like:
      Photos\<Name>\file1.jpg,Photos\<Name>\file2.png
    If folder doesn't exist or name is blank -> return empty string.
    """
    if pd.isna(name_in_english):
        return ""
    name = str(name_in_english).strip()
    if not name:
        return ""
    person_dir = PHOTOS_DIR / name
    if not person_dir.exists() or not person_dir.is_dir():
        return ""
    # Collect image files in that subfolder (top-level only)
    files = sorted(
        [p for p in person_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTS],
        key=lambda p: p.name.lower()
    )
    # Build relative paths using backslashes, as you want: Photos\<Name>\file.ext
    rel_paths = [fr"Photos\{name}\{p.name}" for p in files]
    return ",".join(rel_paths)

# Create/overwrite the Photos column
df["Photos"] = df["Name in English"].apply(build_photos_cell)

#--------- Build Latitude Longitude
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
# ----------------------------
# CONFIG
# ----------------------------
df['Country'] = "Iran"


PROJECT_ROOT = Path(r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP")

CACHE_JSON_PATH = PROJECT_ROOT / "geocode_cache_iran.json"     # persistent cache
CACHE_CSV_PATH  = PROJECT_ROOT / "geocode_cache_iran.csv"      # optional readable cache
DEFAULTED_LOG   = PROJECT_ROOT / "geocode_defaulted_rows.csv"  # optional

DEFAULT_LAT = 33.3
DEFAULT_LON = 55.5

# Nominatim (OSM) geocoder
geolocator = Nominatim(
    user_agent="price_of_liberty_map (contact: sanazgolriz@gmail.com)",
    timeout=10
)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=2,
    error_wait_seconds=10
)

# ----------------------------
# CACHE I/O
# ----------------------------
def load_cache_json(path: Path) -> dict:
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_cache_json(cache: dict, path: Path) -> None:
    path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")

def save_cache_csv(cache: dict, path: Path) -> None:
    """
    Saves the cache as a CSV with columns: query, latitude, longitude
    """
    rows = []
    for q, (lat, lon) in cache.items():
        rows.append({"query": q, "latitude": lat, "longitude": lon})
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")

CACHE = load_cache_json(CACHE_JSON_PATH)

# ----------------------------
# NORMALIZATION
# ----------------------------
def norm(s) -> str:
    if pd.isna(s) or s is None:
        return ""
    s = str(s).strip()
    return " ".join(s.split())

def cache_key(query: str) -> str:
    return query.lower().strip()

def make_queries(city: str, prov: str, country: str):
    """
    Generates the query strings we will use for geocoding/caching.
    """
    out = []
    if city and prov:
        out.append(("city+province", f"{city}, {prov}, {country}"))
    if city:
        out.append(("city", f"{city}, {country}"))
    if prov:
        out.append(("province_fallback", f"{prov}, {country}"))
    return out

# ----------------------------
# SEED THE CACHE FROM EXISTING df LAT/LON (BEFORE YOU DROP THEM)
# ----------------------------
def seed_cache_from_df(df: pd.DataFrame, cache: dict) -> dict:
    """
    If df has existing 'latitude'/'longitude', add them into cache for the row’s query variants.
    This way, after you drop lat/lon, you can still recover them from cache without re-geocoding.
    """
    has_latlon = ("latitude" in df.columns) and ("longitude" in df.columns)
    if not has_latlon:
        return cache
    for _, row in df.iterrows():
        lat = row.get("latitude")
        lon = row.get("longitude")
        if pd.isna(lat) or pd.isna(lon):
            continue
        try:
            lat = float(lat)
            lon = float(lon)
        except Exception:
            continue
        city = norm(row.get("City"))
        prov = norm(row.get("Province"))
        country = norm(row.get("Country")) or "Iran"

        # add to cache for all applicable query variants
        for _, q in make_queries(city, prov, country):
            k = cache_key(q)
            # Only set if not already set, or if it was None/None
            if k not in cache or cache[k] == [None, None] or cache[k] == (None, None):
                cache[k] = [lat, lon]
    return cache

# ----------------------------
# GEOCODE WITH CACHE FIRST
# ----------------------------
def geocode_query(query: str):
    """
    Returns (lat, lon) or (None, None).
    Uses persistent cache to avoid re-querying.
    """
    k = cache_key(query)
    if k in CACHE:
        lat, lon = CACHE[k]
        if lat is None or lon is None:
            return None, None
        return float(lat), float(lon)
    loc = geocode(query)
    if loc is None:
        CACHE[k] = [None, None]
        return None, None
    lat_lon = (float(loc.latitude), float(loc.longitude))
    CACHE[k] = [lat_lon[0], lat_lon[1]]
    return lat_lon

def get_lat_lon_smart(row):
    """
    Uses ONLY City/Province/Country + cache + geocoding.
    Does NOT read any existing df lat/lon (since you'll drop them).
    """
    city = norm(row.get("City"))
    prov = norm(row.get("Province"))
    country = norm(row.get("Country")) or "Iran"

    # If both missing -> default immediately
    if not city and not prov:
        return DEFAULT_LAT, DEFAULT_LON, "default_no_city_no_province", "manual_default"

    attempts = make_queries(city, prov, country)

    for level, q in attempts:
        lat, lon = geocode_query(q)
        if lat is not None and lon is not None:
            return lat, lon, level, q
    # If all attempts fail -> default
    return DEFAULT_LAT, DEFAULT_LON, "default_all_methods_failed", "manual_default"
# ----------------------------
# RUN: seed cache -> drop lat/lon -> recompute -> save cache
# ----------------------------
# 1) Seed cache from any existing lat/lon columns (if present)
CACHE = seed_cache_from_df(df, CACHE)
# 2) Save cache immediately so you don't lose seeded values
save_cache_json(CACHE, CACHE_JSON_PATH)
save_cache_csv(CACHE, CACHE_CSV_PATH)
# 3) Drop existing latitude/longitude columns (if they exist)
df = df.drop(columns=[c for c in ["latitude", "longitude"] if c in df.columns])
# 4) Recompute latitude/longitude fresh (cache-first, then geocode, then default)
res = df.apply(get_lat_lon_smart, axis=1, result_type="expand")
res.columns = ["latitude", "longitude", "geo_match_level", "geo_query_used"]
df[["latitude", "longitude", "geo_match_level", "geo_query_used"]] = res
# 5) Save cache again (it may have grown)
save_cache_json(CACHE, CACHE_JSON_PATH)
save_cache_csv(CACHE, CACHE_CSV_PATH)
# 6) Optional: log rows that were defaulted
defaulted = df[df["geo_match_level"].str.startswith("default", na=False)].copy()
if not defaulted.empty:
    defaulted.to_csv(DEFAULTED_LOG, index=False, encoding="utf-8-sig")
print("Done.")
print(f"Cache JSON: {CACHE_JSON_PATH} (entries: {len(CACHE)})")
print(f"Cache CSV : {CACHE_CSV_PATH}")
if not defaulted.empty:
    print(f"Defaulted rows logged to: {DEFAULTED_LOG} (count: {len(defaulted)})")

# -------- helpers --------
def to_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None

# Find Source1, Source2, ... columns
source_cols = [c for c in df.columns if re.match(r"Source\d+", str(
    c))]
source_cols = sorted(source_cols, key=lambda c: int(re.findall(r"\d+", c)[0])) if source_cols else []
link_col  = "Links"  if "Links"  in df.columns else None
photo_col = "Photos" if "Photos" in df.columns else None

# -------- build GeoJSON --------
features = []
for _, r in df.iterrows():
    lat = r.get("latitude")
    lon = r.get("longitude")
    if pd.isna(lat) or pd.isna(lon):
        continue
    sources = [to_str(r.get(c)) for c in source_cols]
    sources = [s for s in sources if s]
    links_raw = to_str(r.get(link_col)) if link_col else None
    links = []
    if links_raw:
        parts = re.split(r"[\n,]+", links_raw)
        links = [p.strip() for p in parts if p.strip()]
    photos_raw = to_str(r.get(photo_col)) if photo_col else None
    photos = []
    if photos_raw:
        parts = re.split(r"[\n,]+", photos_raw)
        for p in parts:
            p = p.strip()
            if not p:
                continue
            p = p.replace("\\", "/")
            # If they wrote "Photos/xxx.png" keep only filename because JS does "Photos/<filename>"
            if p.lower().startswith("photos/"):
                p = p.split("/", 1)[1].strip()
            photos.append(p)
    g = to_str(r.get("Gender"))
    if g:
        g = g.strip().lower()
    props = {
        "name_fa": to_str(r.get("Name in Farsi")),
        "name_en": to_str(r.get("Name in English")),
        "year": None if pd.isna(r.get("Year")) else int(r.get("Year")),
        "age": None if pd.isna(r.get("Age")) else int(r.get("Age")),
        "gender": g,
        "city": to_str(r.get("City")),
        "province": to_str(r.get("Province")),
        "country": to_str(r.get("country")) or "Iran",
        "sources": sources,
        "links": links,
        "photos": photos,   
    }
    features.append({
        "type": "Feature",
        "properties": props,
        "geometry": {"type": "Point", "coordinates": [float(lon), float(lat)]}  # GeoJSON order: [lon, lat]
    })

geojson = {"type": "FeatureCollection", "features": features}

out_geo = os.path.join(site_dir, "people.geojson")
with open(out_geo, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)

# -------- website files --------
index_html = """<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>The Price of Liberty – Map</title>

  <!-- Leaflet -->
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"
        integrity="sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=" crossorigin=""/>

  <!-- MarkerCluster -->
  <link rel="stylesheet" href="https://unpkg.com/leaflet.markercluster@1.5.3/dist/MarkerCluster.css" />
  <link rel="stylesheet" href="https://unpkg.com/leaflet.markercluster@1.5.3/dist/MarkerCluster.Default.css" />

  <!-- noUiSlider (two-handle range slider) -->
  <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/nouislider@15.7.1/dist/nouislider.min.css">

  <link rel="stylesheet" href="styles.css" />
</head>
<body>
  <div id="app">
    <aside id="sidebar">
     <button id="sidebarToggle" class="sidebar-toggle" type="button" aria-expanded="true">
       ☰
     </button>
      <div class="brand">The Price of Liberty بهای آزادی</div>
      
      <label class="label">Search</label>
      <input id="search" type="search" placeholder="Search name / city / province…" />

      <div class="section">
        <div class="section-title">Year / سال </div>
        <div id="yearRange"></div>
        <div class="yearrow">
          <span id="yearLabel">Years: —</span>
        </div>
      </div>
      
     <div class="section">
       <div class="section-title">Age / سن </div>
       <div id="ageRange"></div>
       <div class="yearrow">
         <span id="ageLabel">Ages: —</span>
       </div>
     </div>
     
     <div class="section">
      <div class="section-title">Gender / جنسیت</div>
      <select id="genderFilter">
        <option value="all">All / همه </option>
        <option value="male">Male / مرد </option>
        <option value="female">Female / زن </option>
      </select>
    </div>
    
      <div class="section">
        <div class="section-title">Shown</div>
        <div id="counter">—</div>
      </div>

      <div class="hint">
        Tip: Click a tulip marker to open the person card.<br>
        بر روی لاله کلیک کنید تا اطلاعات مربوط به هر جاویدنام را ببینید  
      </div> 
       
     <div class="section">
      <a class="submit-link"
         href="https://docs.google.com/forms/d/e/1FAIpQLSd2pQm8CVvGX2QCjmMZrHyUuk9ROqgthd2sabhVLVdkBLPIJA/viewform?usp=dialog"
         target="_blank"
         rel="noopener noreferrer">
        Add a Name<br>
        افزودن نام ↗
      </a>

      <a class="sheet-link"
         href="https://docs.google.com/spreadsheets/d/19IE4FLYkJ1MAvawhhZx5WEnj0xowtOIoqPIB-E-6rSU/edit?usp=sharing"
         target="_blank"
         rel="noopener noreferrer">
        The Memorial Archive<br>
    آرشیو اسامی جاویدنامان ↗
      </a>
      <div class="hint" style="margin-top:2px;">
        This link opens a read-only sheet in a new tab.
       </div>
      </div>

   
    </aside>

    <main id="mapwrap">
      <div id="map"></div>
    </main>
  </div>

  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"
          integrity="sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=" crossorigin=""></script>

  <script src="https://unpkg.com/leaflet.markercluster@1.5.3/dist/leaflet.markercluster.js"></script>

  <script src="https://cdn.jsdelivr.net/npm/nouislider@15.7.1/dist/nouislider.min.js"></script>

  <script src="script.js"></script>
</body>
</html>
"""

styles_css = """html, body { height: 100%; margin: 0; }
body {
  font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif;
  background: #0b0f14;
  color: #e6edf3;
}

#app {
  height: 100%;
  display: flex;
}
.photo-controls{
  display:flex;
  align-items:center;
  justify-content:space-between;
  gap:10px;
  margin-top:10px;
}

.photo-btn{
  padding:8px 10px;
  border-radius:10px;
  border:1px solid rgba(255,255,255,0.14);
  background: rgba(255,255,255,0.06);
  color:#e6edf3;
  cursor:pointer;
  font-weight:700;
  font-size:12px;
}
.photo-btn:disabled{
  opacity:.45;
  cursor:not-allowed;
}
.photo-page{
  font-size:12px;
  opacity:.85;
  white-space:nowrap;
}


.submit-link{
  display: block;
  width: 100%;
  box-sizing: border-box;

  margin-top: 10px;
  padding: 14px 16px;

  border-radius: 16px;
  border: 1px solid rgba(255,255,255,0.12);

  background: linear-gradient(
    145deg,
    rgba(255,255,255,0.06),
    rgba(255,255,255,0.03)
  );

  color: #7dd3fc;
  font-weight: 800;
  text-decoration: none;
  text-align: center;

  transition: all 0.2s ease;
}

.submit-link:hover{
  background: rgba(255,255,255,0.10);
  border-color: rgba(255,255,255,0.18);
}
/* --- Sidebar collapse toggle --- */
.sidebar-toggle{
  display: inline-flex;
  align-items: center;
  justify-content: center;

  width: 38px;
  height: 38px;
  border-radius: 12px;

  border: 1px solid rgba(255,255,255,0.14);
  background: rgba(255,255,255,0.06);
  color: #e6edf3;

  cursor: pointer;
  font-weight: 900;
  margin-bottom: 10px;
}

/* Smooth animate */
#sidebar{
  transition: width 220ms ease, min-width 220ms ease, max-width 220ms ease, padding 220ms ease;
}

/* Collapsed state (desktop/tablet) */
#app.sidebar-collapsed #sidebar{
  width: 56px;
  min-width: 56px;
  max-width: 56px;
  padding: 14px 8px;
}


#app.sidebar-collapsed #sidebar .sidebar-toggle{
  display: inline-flex;
  margin-bottom: 0;
}

#app.sidebar-collapsed #sidebar .brand,
#app.sidebar-collapsed #sidebar .label,
#app.sidebar-collapsed #sidebar #search,
#app.sidebar-collapsed #sidebar .section,
#app.sidebar-collapsed #sidebar .hint{
  display: none;
}

#sidebar {
  width: 260px;
  min-width: 260px;
  max-width: 260px;
  padding: 14px 14px;
  border-right: 1px solid rgba(255,255,255,0.10);
  background: rgba(10, 14, 20, 0.92);
  backdrop-filter: blur(10px);
  box-shadow: 12px 0 30px rgba(0,0,0,0.35);
  z-index: 900; /* below Leaflet controls/popup */
  overflow-y: auto;
}

.sheet-link{
  display: block;
  width: 100%;
  box-sizing: border-box;

  margin-top: 10px;
  padding: 14px 16px;

  border-radius: 16px;
  border: 1px solid rgba(255,255,255,0.12);

  background: linear-gradient(
    145deg,
    rgba(255,255,255,0.06),
    rgba(255,255,255,0.03)
  );

  color: #7dd3fc;
  font-weight: 800;
  text-decoration: none;
  text-align: center;

  transition: all 0.2s ease;
}

.sheet-link:hover{
  background: rgba(255,255,255,0.10);
  border-color: rgba(255,255,255,0.18);
}

.brand {
  font-weight: 900;
  letter-spacing: 0.2px;
  font-size: 16px;
  margin-bottom: 12px;
}

.label {
  display: block;
  font-size: 12px;
  opacity: 0.8;
  margin: 8px 0 6px;
}

#search {
  width: 100%;
  padding: 10px 12px;
  border-radius: 12px;
  border: 1px solid rgba(255,255,255,0.12);
  background: rgba(255,255,255,0.06);
  color: #e6edf3;
  outline: none;
}
#search::placeholder { color: rgba(230,237,243,0.55); }

#genderFilter{
  width: 100%;
  padding: 10px 12px;
  border-radius: 12px;
  border: 1px solid rgba(255,255,255,0.12);
  background: rgba(255,255,255,0.06);
  color: #e6edf3;
  outline: none;
  font-weight: 600;
  cursor: pointer;
}

.section {
  margin-top: 14px;
  padding-top: 12px;
  border-top: 1px solid rgba(255,255,255,0.05);
}

.section-title {
  font-size: 12px;
  opacity: 0.85;
  margin-bottom: 8px;
}

#yearRange{
  margin: 40px auto 0 auto;
  width: 90%;
  max-width: 100%;
  padding: 0 1px;   /* THIS creates breathing room */
  box-sizing: border-box;
}
.yearrow {
  display: flex;
  justify-content: space-between;
  font-size: 12px;
  opacity: 0.85;
  margin-top: 8px;
}

#ageRange {margin: 40px auto 0 auto;
  width: 90%;
  max-width: 100%;
  padding: 0 1px;   /* THIS creates breathing room */
  box-sizing: border-box;}

#counter {
  font-size: 18px;
  font-weight: 800;
  margin-top: 4px;
}

.hint {
  margin-top: 15px;
  font-size: 12px;
  opacity: 0.7;
  line-height: 1.4;
}

#mapwrap { flex: 1; position: relative; }
#map { height: 100%; width: 100%; }

/* Leaflet popup look (keep photobook CSS additions below if you already have them) */
.leaflet-popup-content-wrapper {
  background: rgba(12, 16, 22, 0.95);
  color: #e6edf3;
  border-radius: 14px;
  border: 1px solid rgba(255,255,255,0.10);
}
.leaflet-popup-tip { background: rgba(12, 16, 22, 0.95); }

.popup-title { font-weight: 800; font-size: 16px; margin-bottom: 4px; }
.popup-sub { opacity: 0.85; margin-bottom: 8px; }
.popup-links a { color: #7dd3fc; text-decoration: none; }
.popup-links a:hover { text-decoration: underline; }

/* ✅ NEW: make sure photos fit inside popup nicely (won’t break existing photobook) */
.photobook .photo-frame { width: 100%; overflow: hidden; border-radius: 12px; }
.photobook .photo-img { width: 100%; height: auto; display: block; object-fit: contain; }

.photobook .photo-frame{
  position: relative;
  overflow:hidden;
  border-radius:12px;
}

.photobook .edge-flash{
  position:absolute;
  inset:0;
  pointer-events:none;
  border-radius:12px;
  opacity:0;
  background: linear-gradient(to left,
    rgba(255,255,255,0.00) 0%,
    rgba(255,255,255,0.10) 70%,
    rgba(255,255,255,0.00) 100%);
  transform: translateX(0);
}
.photobook.flash .edge-flash{
  opacity:1;
  transition: opacity 140ms ease;
}


/* noUiSlider styling to match dark UI */
.noUi-target { border: 1px solid rgba(255,255,255,0.15); background: rgba(255,255,255,0.06); box-shadow: none; }
.noUi-connect { background: rgba(125,211,252,0.35); }
.noUi-handle { background: rgba(230,237,243,0.9); border-radius: 10px; box-shadow: none; border: none; }
.noUi-tooltip { background: rgba(12,16,22,0.95); color: #e6edf3; border: 1px solid rgba(255,255,255,0.12); }
.noUi-tooltip{
  font-size: 11px;
  padding: 3px 6px;
  border-radius: 8px;
  white-space: nowrap;
}

.noUi-handle{
  width: 18px !important;
  height: 18px !important;
  right: -9px !important; /* keeps handle centered */
  top: -7px !important;
  border-radius: 10px;
}
.noUi-horizontal .noUi-tooltip{transform: translate(-50%, 10%);}
.noUi-horizontal .noUi-handle-lower .noUi-tooltip{left: 50%;}
.noUi-horizontal .noUi-handle-upper .noUi-tooltip{left: 50%;}

/* Optional: on small screens, sidebar becomes bottom drawer */
@media (max-width: 900px) {
  #app { flex-direction: column; }
  #sidebar { width: 100%; min-width: 100%; max-width: 100%; border-right: none; border-top: 1px solid rgba(255,255,255,0.10); }
}
@media (max-width: 900px) {
  /* On mobile, collapsed = hide content, but keep the bar */
  #app.sidebar-collapsed #sidebar{
    width: 100%;
    min-width: 100%;
    max-width: 100%;
    padding: 10px 14px;
  }
  #app.sidebar-collapsed #sidebar > *:not(.sidebar-toggle){
    display: none;
  }
}

"""

script_js = """// Dark tiles (free): CARTO 'Dark Matter'
const darkTiles = L.tileLayer(
  'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
  {
    attribution: '&copy; OpenStreetMap contributors &copy; CARTO',
    subdomains: 'abcd',
    maxZoom: 19
  }
);

// Center on Iran
const map = L.map('map', { layers: [darkTiles], zoomControl: true }).setView([32.5, 53.7], 5);

// Move zoom buttons away from sidebar overlap (optional)
map.zoomControl.setPosition('bottomright');

// Rough bounds for Iran
const iranBounds = L.latLngBounds(L.latLng(24.8, 44.0), L.latLng(39.9, 63.5));
map.setMaxBounds(iranBounds.pad(0.15));
map.on('drag', () => map.panInsideBounds(iranBounds, { animate: false }));

// Marker cluster
const cluster = L.markerClusterGroup({ showCoverageOnHover: false, maxClusterRadius: 45 });

let allMarkers = [];
let allFeatures = [];

// Range filter values
let minYear = -Infinity;
let maxYear = Infinity;

let minAge = -Infinity;
let maxAge = Infinity;

// UI elements (same IDs, just in sidebar now)
const yearLabel = document.getElementById('yearLabel');
const yearRangeEl = document.getElementById('yearRange');

const ageLabel = document.getElementById('ageLabel');
const ageRangeEl = document.getElementById('ageRange');

const counter = document.getElementById('counter');
const searchBox = document.getElementById('search');
const genderFilter = document.getElementById('genderFilter');

function escapeHtml(s) {
  return String(s ?? '')
    .replaceAll('&', '&amp;')
    .replaceAll('<', '&lt;')
    .replaceAll('>', '&gt;')
    .replaceAll('\"', '&quot;')
    .replaceAll("'", '&#039;');
}

/* =========================================================
   PHOTOBOOK popupHtml(p) FUNCTION HERE.
   ========================================================= */
function popupHtml(p) {
  const name = [p.name_en, p.name_fa].filter(Boolean).join(' / ');
  const loc  = [p.city, p.province].filter(Boolean).join(', ');
  const yr   = (p.year !== null && p.year !== undefined) ? `Year: ${p.year}` : '';
  const age  = (p.age  !== null && p.age  !== undefined) ? `Age: ${p.age}` : '';
  const meta = [loc, yr, age].filter(Boolean).join(' • ');

  // Photos array -> build a little “photobook”
  const photos = Array.isArray(p.photos) ? p.photos.filter(Boolean) : [];
  const photoCount = photos.length;

  const photoHtml = photoCount
    ? `
      <div class="photobook" data-photo-index="0" data-photo-total="${photoCount}">
        <div class="photo-frame">
          <img class="photo-img" src="Photos/${escapeHtml(photos[0])}" alt="photo" loading="lazy"
               onerror="this.style.display='none'; this.parentElement.classList.add('photo-missing');" />
        </div>

        <div class="photo-controls">
          <button class="photo-btn prev" type="button" ${photoCount <= 1 ? "disabled" : ""}>Previous</button>
          <div class="photo-page">${photoCount > 0 ? `1 / ${photoCount}` : ""}</div>
          <button class="photo-btn next" type="button" ${photoCount <= 1 ? "disabled" : ""}>Next</button>
        </div>
      </div>
    `
    : `<div class="photobook photobook-empty">No photos yet.</div>`;

  // Links (Sources + Links)
  const links = [];
  const sources = Array.isArray(p.sources) ? p.sources : [];
  const extras  = Array.isArray(p.links)   ? p.links   : [];

  for (const url of sources) links.push(url);
  for (const url of extras)  links.push(url);

  const linksHtml = links.length
    ? `<div class="popup-links">${links.map((u, i) => {
        const safe = escapeHtml(u);
        const label = (safe.includes('youtube.com') || safe.includes('youtu.be')) ? `Video ${i+1}` : `Link ${i+1}`;
        return `<div><a href="${safe}" target="_blank" rel="noopener noreferrer">${label}</a></div>`;
      }).join('')}</div>`
    : `<div class="popup-links" style="opacity:.75;">No links yet.</div>`;

  return `
    <div class="popup-title">${escapeHtml(name || 'Unknown')}</div>
    <div class="popup-sub">${escapeHtml(meta)}</div>

    ${photoHtml}

    ${linksHtml}
  `;
}


function wirePhotoBook(popupEl, photos) {
  if (!popupEl) return;
  const wrap = popupEl.querySelector('.photobook');
  if (!wrap) return;

  const img = wrap.querySelector('.photo-img');
  const page = wrap.querySelector('.photo-page');
  const prev = wrap.querySelector('.photo-btn.prev');
  const next = wrap.querySelector('.photo-btn.next');

  if (!img || !page || !prev || !next) return;

  let idx = 0;
  const total = photos.length;

  function render() {
    img.style.display = '';
    img.src = `Photos/${photos[idx]}`;
    page.textContent = `${idx + 1} / ${total}`;
    prev.disabled = total <= 1;
    next.disabled = total <= 1;
  }

  // Important: avoid double-wiring if popup opens multiple times
  prev.onclick = (e) => {
    e.preventDefault();
    e.stopPropagation();
    idx = (idx - 1 + total) % total;
    render();
  };

  next.onclick = (e) => {
    e.preventDefault();
    e.stopPropagation();
    idx = (idx + 1) % total;
    render();
  };

  render();
}


function makeMarker(feature) {
  const [lon, lat] = feature.geometry.coordinates;
  const p = feature.properties || {};

  const icon = L.icon({
    iconUrl: 'tulip_icon.svg',
    iconSize: [32, 32],
    iconAnchor: [16, 32],
    // keep existing popup positioning
    popupAnchor: [0, -30]
  });

  const m = L.marker([lat, lon], { icon }).bindPopup(popupHtml(p), {
    maxWidth: 360,
    closeButton: true,
    autoPan: true,
    keepInView: true
  });

  //attach photobook button wiring when popup opens
  m.on('popupopen', (e) => {
    const photos = Array.isArray(p.photos) ? p.photos.filter(Boolean) : [];
    if (!photos.length) return;
    const popupEl = e.popup.getElement();
    wirePhotoBook(popupEl, photos);
  });

  m.__props = p;
  return m;
}

function matchesSearch(p, q) {
  if (!q) return true;
  const hay = [p.name_en, p.name_fa, p.city, p.province].filter(Boolean).join(' ').toLowerCase();
  return hay.includes(q);
}

function matchesYear(p) {
  if (p.year === null || p.year === undefined) return true;
  const y = Number(p.year);
  return y >= minYear && y <= maxYear;
}

function matchesAge(p) {
  // If missing age, SHOW it (same behavior as year)
  if (p.age === null || p.age === undefined) return true;
  const a = Number(p.age);
  return a >= minAge && a <= maxAge;
}

function matchesGender(p){
  const selected = genderFilter.value;
  if (selected === "all") return true;
  if (!p.gender) return false;  // hide if no gender when filtering
  return p.gender === selected;
}

function applyFilter() {
  const q = searchBox.value.trim().toLowerCase();
  const filtered = allMarkers.filter(m => {
    const p = m.__props || {};
    return matchesYear(p)
    && matchesAge(p)
    && matchesGender(p)
    && matchesSearch(p, q);
  });

  cluster.clearLayers();
  cluster.addLayers(filtered);
  counter.textContent = `${filtered.length} shown`;
}

async function loadData() {
  const res = await fetch('people.geojson');
  if (!res.ok) throw new Error(`Failed to load people.geojson (${res.status})`);
  const gj = await res.json();

  allFeatures = gj.features || [];
  allMarkers = allFeatures.map(makeMarker);

  cluster.clearLayers();
  cluster.addLayers(allMarkers);
  map.addLayer(cluster);

  // ---------- IRAN OUTLINE ----------
  fetch('iran_boundary.geojson')
    .then(res => res.json())
    .then(data => {
      L.geoJSON(data, {
        style: {
          color: "#50C878",
          weight: 2,
          opacity: 0.9,
          fillOpacity: 0
        }
      }).addTo(map);
    })
    .catch(err => console.error("Iran boundary load error:", err));

  // Build range slider dynamically from data years
  const years = allFeatures
    .map(f => f.properties?.year)
    .filter(y => y !== null && y !== undefined && !Number.isNaN(Number(y)))
    .map(y => Number(y));

  if (years.length) {
    const minY = Math.min(...years);
    const maxY = Math.max(...years);

    noUiSlider.create(yearRangeEl, {
      start: [minY, maxY],
      connect: true,
      step: 1,
      range: { min: minY, max: maxY },
      tooltips: [true, true],
      format: {
        to: v => Math.round(v),
        from: v => Number(v)
      }
    });

    minYear = minY;
    maxYear = maxY;
    yearLabel.textContent = `Years: ${minYear} – ${maxYear}`;

    yearRangeEl.noUiSlider.on('update', (values) => {
      minYear = Number(values[0]);
      maxYear = Number(values[1]);
      yearLabel.textContent = `Years: ${minYear} – ${maxYear}`;
      applyFilter();
    });
  } else {
    yearLabel.textContent = "Years: —";
  }
  // Build age slider dynamically from data ages
  const ages = allFeatures
    .map(f => f.properties?.age)
    .filter(a => a !== null && a !== undefined && !Number.isNaN(Number(a)))
    .map(a => Number(a));

  if (ages.length) {
    const minA = Math.min(...ages);
    const maxA = Math.max(...ages);

    noUiSlider.create(ageRangeEl, {
      start: [minA, maxA],
      connect: true,
      step: 1,
      range: { min: minA, max: maxA },
      tooltips: [true, true],
      format: {
        to: v => Math.round(v),
        from: v => Number(v)
      }
    });

    minAge = minA;
    maxAge = maxA;
    ageLabel.textContent = `Ages: ${minAge} – ${maxAge}`;

    ageRangeEl.noUiSlider.on('update', (values) => {
      minAge = Number(values[0]);
      maxAge = Number(values[1]);
      ageLabel.textContent = `Ages: ${minAge} – ${maxAge} Years Old`;
      applyFilter();
    });
  } else {
    ageLabel.textContent = "Ages: —";
  }

  counter.textContent = `${allMarkers.length} points`;
  applyFilter();
}

// Events
searchBox.addEventListener('input', () => {
  window.clearTimeout(window.__t);
  window.__t = window.setTimeout(applyFilter, 120);
});

genderFilter.addEventListener('change', () => {
  applyFilter();
});

loadData().catch(err => {
  console.error(err);
  alert(err.message);
});

// --- Sidebar collapsibility ---
(() => {
  const app = document.getElementById('app');
  const btn = document.getElementById('sidebarToggle');
  if (!app || !btn) return;

  function setExpanded(isExpanded) {
    btn.setAttribute('aria-expanded', String(isExpanded));
    btn.title = isExpanded ? "Collapse" : "Expand";
  }

  // Optional: remember choice
  const saved = localStorage.getItem('sidebarCollapsed');
  if (saved === '1') {
    app.classList.add('sidebar-collapsed');
    setExpanded(false);
  } else {
    setExpanded(true);
  }

  btn.addEventListener('click', () => {
    const collapsed = app.classList.toggle('sidebar-collapsed');
    localStorage.setItem('sidebarCollapsed', collapsed ? '1' : '0');
    setExpanded(!collapsed);

    // Leaflet needs a size refresh when layout changes
    setTimeout(() => map.invalidateSize(), 260);
  });
})();

"""

readme = """# The Price of Liberty - Iran Interactive Map (Leaflet)

I'm building a memorial archive to document all those who have been brutally killed by the dictatorial Islamic republic regime since 1979. My goal is to make this archive as complete as possible, so none will ever be forgotten. I deeply appreciate everyone's input and help in ensuring that every name and story is remembered.

من دارم یک آرشیو یادبود می‌سازم تا همه‌ی کسانی که از سال ۱۳۵۷ به‌دست رژیم دیکتاتوری جمهوری اسلامی به‌طور وحشیانه کشته شده‌اند، ثبت کنم. هدفم این است که این آرشیو تا حد امکان کامل باشد تا هیچ‌کس فراموش نشود. از کمک و نظرات همه‌ی شما عمیقاً سپاسگزارم تا جاویدنامان به یاد سپرده شود

## What this is
A static site with:
- dark map
- clickable tulips (popups with photobook)
- LEFT collapsible sidebar with searchable information
- marker clustering
- Iran outline overlay

## Files
- index.html
- styles.css
- script.js
- people.geojson
- iran_boundary.geojson
- tulip_icon.svg
- Photos/ (images)

## Run locally
Do NOT double-click index.html.
Run a local server:
cd "C:\\Users\\sanaz\\OneDrive\\Documents\\ThePriceofLibertyMAP"
python -m http.server 8000
Then open:
    http://localhost:8000
"""

# write files
with open(os.path.join(site_dir, "index.html"), "w", encoding="utf-8") as f: f.write(index_html)
with open(os.path.join(site_dir, "styles.css"), "w", encoding="utf-8") as f: f.write(styles_css)
with open(os.path.join(site_dir, "script.js"), "w", encoding="utf-8") as f: f.write(script_js)
with open(os.path.join(site_dir, "README.md"), "w", encoding="utf-8") as f: f.write(readme)

'''
# zip it
zip_path = os.path.join(site_dir, "iran_leaflet_site.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path(site_dir).rglob("*"):
        if p.name.lower().endswith((".xlsx", ".zip")):
            continue
        z.write(p, arcname=str(p.relative_to(site_dir)))
'''

import datetime
import re
timestamp = datetime.datetime.utcnow().isoformat()
# remove previous build comment if it exists
index_html = re.sub(r'<!-- build .*? -->', '', index_html)
# add the new one
index_html += f"\n<!-- build {timestamp} -->"
with open(os.path.join(site_dir, "index.html"), "w", encoding="utf-8") as f:
    f.write(index_html)


Done.
Folders created: 894
Rows skipped (blank/invalid): 1

New folders:
 - Amirhesam Khodayari-Fard
 - Sajjad Zeilaei
 - Vahab Mousavi
 - Ahad Ebrahimpour Abdoli
 - Khodadad Shirvani Monfared
 - Shahpour Parvinmehr
 - Mansour Mokhtari
 - Ali (Nejat) Azizi
 - Farshid Mokhtari
 - Ali Karimi Bavolaki
 - Reza Kadyourian
 - Rasoul Kadyourian
 - Amirmohammad Kouhkan
 - Mohammadghasem Rousta
 - Fars (Mohsen) Aghamohammadi
 - Mobin Yaghoubzadeh
 - Milad Gholamzadeh
 - Rouh-Allah Setareh Moshtari
 - Morteza Jahanbakhsh Varanlou
 - Mehdi Ghorbandoost Varanlou
...
Done.
Cache JSON: C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\geocode_cache_iran.json (entries: 866)
Cache CSV : C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\geocode_cache_iran.csv
Defaulted rows logged to: C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\geocode_defaulted_rows.csv (count: 467)


### Further Processing

In [21]:
import pandas as pd, json, os, re, zipfile, pathlib
excel_path = r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\ListOfNames.xlsx"
site_dir   = r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP"
os.makedirs(site_dir, exist_ok=True)

googlesheet = 'https://docs.google.com/spreadsheets/d/19IE4FLYkJ1MAvawhhZx5WEnj0xowtOIoqPIB-E-6rSU/edit?usp=sharing'
sheet_id = "19IE4FLYkJ1MAvawhhZx5WEnj0xowtOIoqPIB-E-6rSU"
gid = 0  # change if tab isn't the first one
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df = pd.read_csv(url)

df['Name in Farsi_clean'] = df['Name in Farsi'].str.strip()

print(df.shape)

(3548, 15)


#### Image transfers

In [22]:
# create folders and put related filename in respective folders. 
import os
import shutil
from pathlib import Path
PHOTOS_DIR = Path(r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP\Photos")
# Image extensions to include
EXTS = {".png", ".jpg", ".jpeg"}
def safe_folder_name(name: str) -> str:
    """
    Make a Windows-safe folder name.
    Removes characters Windows doesn't allow in file/folder names.
    """
    invalid = '<>:"/\\|?*'
    for ch in invalid:
        name = name.replace(ch, "")
    return name.strip().rstrip(".")  # no trailing dots in Windows

def main():
    if not PHOTOS_DIR.exists():
        raise FileNotFoundError(f"Folder not found: {PHOTOS_DIR}")
    # Find image files in the top-level of PHOTOS_DIR
    image_files = [
        p for p in PHOTOS_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in EXTS
    ]
    # Group by base name
    groups = {}
    for p in image_files:
        base = p.stem  # filename without extension
        groups.setdefault(base, []).append(p)
    # Create folder per base name and move files
    for base, files in groups.items():
        folder_name = safe_folder_name(base)
        if not folder_name:
            print(f"Skipping empty/invalid base name for files: {[f.name for f in files]}")
            continue
        dest_dir = PHOTOS_DIR / folder_name
        dest_dir.mkdir(exist_ok=True)
        for src in files:
            dest = dest_dir / src.name
            # Avoid overwriting if somehow same filename exists in destination
            if dest.exists():
                print(f"Already exists, skipping move: {dest}")
                continue
            shutil.move(str(src), str(dest))
            print(f"Moved: {src.name}  ->  {dest_dir.name}\\{src.name}")
    print("\nDone.")

if __name__ == "__main__":
    main()

Moved: Abbas (Hassan) Akbarpour.jpg  ->  Abbas (Hassan) Akbarpour\Abbas (Hassan) Akbarpour.jpg
Moved: Abbas Aminizadeh Sdeh.jpg  ->  Abbas Aminizadeh Sdeh\Abbas Aminizadeh Sdeh.jpg
Moved: Abbas Amirbeigi.jpg  ->  Abbas Amirbeigi\Abbas Amirbeigi.jpg
Moved: Abbas Arezou.jpg  ->  Abbas Arezou\Abbas Arezou.jpg
Moved: Abbas Goodarzi Qashqaei.jpg  ->  Abbas Goodarzi Qashqaei\Abbas Goodarzi Qashqaei.jpg
Moved: Abbas Khadm.jpg  ->  Abbas Khadm\Abbas Khadm.jpg
Moved: Abbas Kshavrz Lfmjani.jpg  ->  Abbas Kshavrz Lfmjani\Abbas Kshavrz Lfmjani.jpg
Moved: Abbas Maleki.jpg  ->  Abbas Maleki\Abbas Maleki.jpg
Moved: Abbas Zeynali.jpg  ->  Abbas Zeynali\Abbas Zeynali.jpg
Moved: Abdollah Adybi.jpg  ->  Abdollah Adybi\Abdollah Adybi.jpg
Moved: Abdollah Hadizadeh.jpg  ->  Abdollah Hadizadeh\Abdollah Hadizadeh.jpg
Moved: Abdyn Khazaei.jpg  ->  Abdyn Khazaei\Abdyn Khazaei.jpg
Moved: Abolfazl (Masih) MyrAyyz.jpg  ->  Abolfazl (Masih) MyrAyyz\Abolfazl (Masih) MyrAyyz.jpg
Moved: Abolfazl (Mehdi) Nejat.jpg  -> 

#### Comparisons

In [ ]:
newdf = pd.read_excel("../../../Downloads/iran_1401_protesters_moatarezan_cleaned.xlsx", sheet_name='فهرست_نام‌ها')
newdf['نام_clean'] = newdf['نام'].str.strip()
newdf.shape

In [ ]:
new = [name for name in newdf['نام'].unique() if name not in df['Name in Farsi'].unique()]
print(len(new))

In [ ]:
import pandas as pd
import re
def normalize_farsi_name(series: pd.Series) -> pd.Series:
    s = series.fillna('').astype(str)
    s = (
        s.str.strip()
         .str.replace('ي', 'ی', regex=False)
         .str.replace('ك', 'ک', regex=False)
         .str.replace('\u200c', ' ', regex=False)   # نیم‌فاصله -> space
         .str.replace(r'\s+', ' ', regex=True)      # collapse spaces
    )
    # remove common titles/prefixes only at the beginning of the name
    prefixes = r'^(سید|سيد)\s*'
    s = s.str.replace(prefixes, '', regex=True)
    return s.str.strip()

df['name_clean'] = normalize_farsi_name(df['Name in Farsi'])
newdf['name_clean'] = normalize_farsi_name(newdf['نام'])

missing_records = newdf[~newdf['name_clean'].isin(df['name_clean'])]
#missing_records = newdf[~newdf['نام'].isin(df['Name in Farsi'])]
missing_records.to_csv("../../../Downloads/missing.csv",index=False)


##### Geo codes 

In [ ]:
import json
import pandas as pd
from pathlib import Path

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ----------------------------
# CONFIG
# ----------------------------
PROJECT_ROOT = Path(r"C:\Users\sanaz\OneDrive\Documents\ThePriceofLibertyMAP")

CACHE_JSON_PATH = PROJECT_ROOT / "geocode_cache_iran.json"     # persistent cache
CACHE_CSV_PATH  = PROJECT_ROOT / "geocode_cache_iran.csv"      # optional readable cache
DEFAULTED_LOG   = PROJECT_ROOT / "geocode_defaulted_rows.csv"  # optional

DEFAULT_LAT = 33.3
DEFAULT_LON = 55.5

# Nominatim (OSM) geocoder
geolocator = Nominatim(
    user_agent="price_of_liberty_map (contact: sanazgolriz@gmail.com)",
    timeout=10
)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=2,
    error_wait_seconds=10
)

# ----------------------------
# CACHE I/O
# ----------------------------
def load_cache_json(path: Path) -> dict:
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def save_cache_json(cache: dict, path: Path) -> None:
    path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")

def save_cache_csv(cache: dict, path: Path) -> None:
    """
    Saves the cache as a CSV with columns: query, latitude, longitude
    """
    rows = []
    for q, (lat, lon) in cache.items():
        rows.append({"query": q, "latitude": lat, "longitude": lon})
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")

CACHE = load_cache_json(CACHE_JSON_PATH)

# ----------------------------
# NORMALIZATION
# ----------------------------
def norm(s) -> str:
    if pd.isna(s) or s is None:
        return ""
    s = str(s).strip()
    return " ".join(s.split())

def cache_key(query: str) -> str:
    return query.lower().strip()

def make_queries(city: str, prov: str, country: str):
    """
    Generates the query strings we will use for geocoding/caching.
    """
    out = []
    if city and prov:
        out.append(("city+province", f"{city}, {prov}, {country}"))
    if city:
        out.append(("city", f"{city}, {country}"))
    if prov:
        out.append(("province_fallback", f"{prov}, {country}"))
    return out

# ----------------------------
# SEED THE CACHE FROM EXISTING df LAT/LON (BEFORE YOU DROP THEM)
# ----------------------------
def seed_cache_from_df(df: pd.DataFrame, cache: dict) -> dict:
    """
    If df has existing 'latitude'/'longitude', add them into cache for the row’s query variants.
    This way, after you drop lat/lon, you can still recover them from cache without re-geocoding.
    """
    has_latlon = ("latitude" in df.columns) and ("longitude" in df.columns)
    if not has_latlon:
        return cache

    for _, row in df.iterrows():
        lat = row.get("latitude")
        lon = row.get("longitude")
        if pd.isna(lat) or pd.isna(lon):
            continue

        try:
            lat = float(lat)
            lon = float(lon)
        except Exception:
            continue

        city = norm(row.get("City"))
        prov = norm(row.get("Province"))
        country = norm(row.get("Country")) or "Iran"

        # add to cache for all applicable query variants
        for _, q in make_queries(city, prov, country):
            k = cache_key(q)
            # Only set if not already set, or if it was None/None
            if k not in cache or cache[k] == [None, None] or cache[k] == (None, None):
                cache[k] = [lat, lon]

    return cache

# ----------------------------
# GEOCODE WITH CACHE FIRST
# ----------------------------
def geocode_query(query: str):
    """
    Returns (lat, lon) or (None, None).
    Uses persistent cache to avoid re-querying.
    """
    k = cache_key(query)

    if k in CACHE:
        lat, lon = CACHE[k]
        if lat is None or lon is None:
            return None, None
        return float(lat), float(lon)

    loc = geocode(query)
    if loc is None:
        CACHE[k] = [None, None]
        return None, None

    lat_lon = (float(loc.latitude), float(loc.longitude))
    CACHE[k] = [lat_lon[0], lat_lon[1]]
    return lat_lon

def get_lat_lon_smart(row):
    """
    Uses ONLY City/Province/Country + cache + geocoding.
    Does NOT read any existing df lat/lon (since you'll drop them).
    """
    city = norm(row.get("City"))
    prov = norm(row.get("Province"))
    country = norm(row.get("Country")) or "Iran"

    # If both missing -> default immediately
    if not city and not prov:
        return DEFAULT_LAT, DEFAULT_LON, "default_no_city_no_province", "manual_default"

    attempts = make_queries(city, prov, country)

    for level, q in attempts:
        lat, lon = geocode_query(q)
        if lat is not None and lon is not None:
            return lat, lon, level, q

    # If all attempts fail -> default
    return DEFAULT_LAT, DEFAULT_LON, "default_all_methods_failed", "manual_default"

# ----------------------------
# RUN: seed cache -> drop lat/lon -> recompute -> save cache
# ----------------------------

# 1) Seed cache from any existing lat/lon columns (if present)
CACHE = seed_cache_from_df(df, CACHE)

# 2) Save cache immediately so you don't lose seeded values
save_cache_json(CACHE, CACHE_JSON_PATH)
save_cache_csv(CACHE, CACHE_CSV_PATH)

# 3) Drop existing latitude/longitude columns (if they exist)
df = df.drop(columns=[c for c in ["latitude", "longitude"] if c in df.columns])

# 4) Recompute latitude/longitude fresh (cache-first, then geocode, then default)
res = df.apply(get_lat_lon_smart, axis=1, result_type="expand")
res.columns = ["latitude", "longitude", "geo_match_level", "geo_query_used"]
df[["latitude", "longitude", "geo_match_level", "geo_query_used"]] = res

# 5) Save cache again (it may have grown)
save_cache_json(CACHE, CACHE_JSON_PATH)
save_cache_csv(CACHE, CACHE_CSV_PATH)

# 6) Optional: log rows that were defaulted
defaulted = df[df["geo_match_level"].str.startswith("default", na=False)].copy()
if not defaulted.empty:
    defaulted.to_csv(DEFAULTED_LOG, index=False, encoding="utf-8-sig")

print("Done.")
print(f"Cache JSON: {CACHE_JSON_PATH} (entries: {len(CACHE)})")
print(f"Cache CSV : {CACHE_CSV_PATH}")
if not defaulted.empty:
    print(f"Defaulted rows logged to: {DEFAULTED_LOG} (count: {len(defaulted)})")